In [ ]:

# ============================================================
# CELL 1 — WT vs CNTNAP2: LOAD ANIMAL-LEVEL LFP DATA
# ============================================================

%pip -q install scipy statsmodels

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path(
    "/content/drive/MyDrive/Reanalysis/"
    "Our_Analysis/lfp/spontaneous"
)

OUT = BASE / "genotype_comparison" / "WT_vs_Cntnap2"
OUT.mkdir(parents=True, exist_ok=True)

wt = pd.read_csv(
    BASE / "WT/processing/WT_LFP_all_animal_results.csv"
)

cnt = pd.read_csv(
    BASE / "Cntnap2/processing/Cntnap2_LFP_all_animal_results.csv"
)

wt["genotype"] = "WT"
cnt["genotype"] = "Cntnap2"

data_all = pd.concat([wt, cnt], ignore_index=True)

BANDS = [
    "Delta", "Theta", "Alpha-like",
    "Beta", "Low gamma", "High gamma"
]

REGIONS = [
    "DG", "CA1", "CA3", "SUB",
    "VISp", "MOs", "CP", "MRN", "APN"
]

MIN_ANIMALS = 5

key = ["genotype", "subject", "region", "subregion"]

assert not data_all.duplicated(key).any(), (
    "Duplicate animal-region observations"
)

# Count independent animals, not channels or sessions.
coverage = (
    data_all.groupby(
        ["region", "subregion", "genotype"]
    )["subject"]
    .nunique()
    .unstack(fill_value=0)
)

for genotype in ["WT", "Cntnap2"]:
    if genotype not in coverage.columns:
        coverage[genotype] = 0

coverage["eligible"] = (
    (coverage["WT"] >= MIN_ANIMALS) &
    (coverage["Cntnap2"] >= MIN_ANIMALS)
)

eligible_groups = coverage.index[coverage["eligible"]]

data = (
    data_all
    .set_index(["region", "subregion"])
    .loc[eligible_groups]
    .reset_index()
)

# Save both full and eligible data.
data_all.to_csv(OUT / "all_animal_data.csv", index=False)
data.to_csv(OUT / "combined_animal_data.csv", index=False)
coverage.to_csv(OUT / "group_coverage.csv")

print("WT animals:", data.loc[
    data.genotype == "WT", "subject"
].nunique())

print("Cntnap2 animals:", data.loc[
    data.genotype == "Cntnap2", "subject"
].nunique())

print("Comparable anatomical groups:", len(eligible_groups))
display(coverage)
